# Notebook 06 — Information Theory Laboratory

Statistical inference asked: *how uncertain is an estimate?* Information theory asks a more primitive question:

> **How much uncertainty is carried by a random variable, how surprising is one outcome, how much does one variable reveal about another, and what does it cost when a model assigns the wrong probabilities?**

This notebook develops two connected chains:

$$
\boxed{
P(x)\rightarrow I(x)\rightarrow H(X)\rightarrow H(X,Y)\rightarrow
H(Y\mid X)\rightarrow I(X;Y)\rightarrow \text{information gain}
}
$$

and

$$
\boxed{
P\text{ (reality)},Q\text{ (model)}
\rightarrow -\log Q(x)
\rightarrow H(P,Q)
\rightarrow D_{KL}(P\parallel Q)
\rightarrow \text{classification loss}
}
$$

The logarithm will appear in both chains for the same reason: independent probabilities multiply, while information and loss should add.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from IPython.display import display
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import log_loss, mutual_info_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.linear_algebra_and_stat import information_theory as it
from src.linear_algebra_and_stat import information_theory_visualizations as itv

DATA_DIR = PROJECT_ROOT / "data" / "information_theory"
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.dpi"] = 110
RNG = np.random.default_rng(42)

## 1. From Probability to Surprise

Before entropy, consider one observed outcome $x$.

* If $P(x)=0.99$, its arrival barely changes what we expected.
* If $P(x)=0.01$, its arrival is surprising and should carry more information.
* If $P(x)=1$, learning that it happened tells us nothing.

So an information measure $I(p)$ should satisfy:

1. $I(p)$ decreases as $p$ increases.
2. $I(1)=0$.
3. Independent events add information. If $P(x,y)=P(x)P(y)$, then
   $$I(P(x)P(y))=I(P(x))+I(P(y)).$$

The continuous solutions to the functional equation $I(pq)=I(p)+I(q)$ are logarithms:

$$I(p)=-k\log p.$$

The minus sign makes information non-negative because $\log p\leq0$ on $0<p\leq1$. Choosing $k$ or, equivalently, the log base determines the unit:

$$
\boxed{I(x)=-\log_b P(x)}
$$

* $b=2$: **bits**
* $b=e$: **nats**
* $b=10$: **hartleys** (less common)

> **Notation warning:** $I(x)$ here is self-information in one outcome. Later,
> $I(X;Y)$ will denote mutual information between two random variables. The
> shared letter is conventional, but these are different mathematical objects.

For $P(x)=1/8$:

$$I(x)=-\log_2(1/8)=3\text{ bits}.$$

Why exactly three? Three yes/no answers can distinguish $2^3=8$ equally likely possibilities. Information in bits is not merely a metaphor; it is a description-length scale.

In [ ]:
# From-scratch self-information, checked against direct logarithms.
for p in [1.0, 0.5, 0.125, 0.01]:
    manual = it.self_information(p, base=2)
    direct = -np.log2(p)
    print(f"P(x)={p:>6}: I(x)={manual:7.3f} bits  direct={direct:7.3f}")

itv.plot_probability_vs_information()

The curve diverges as $p\to0$: a model that says an event is nearly impossible receives an enormous surprise when it occurs. That same divergence will become cross-entropy loss.

### Why independent information adds

For independent $x$ and $y$:

$$
\begin{aligned}
I(x,y)
&=-\log_2 P(x,y)\\
&=-\log_2[P(x)P(y)]\\
&=-\log_2P(x)-\log_2P(y)\\
&=I(x)+I(y).
\end{aligned}
$$

This conversion of multiplication into addition is the mathematical reason for the logarithm.

## 2. Actual Bits, Repetition, and Average Description Length

Suppose a source emits $0$ or $1$.

* A fair source has $P(1)=P(0)=1/2$. Each result carries one bit.
* A repetitive source with $P(1)=0.99$ usually says the same thing. The common `1` carries only $-\log_2(0.99)\approx0.0145$ bits, while the rare `0` carries $6.64$ bits.

You cannot generally assign a fractional number of bits to one isolated symbol with an ordinary binary code. The meaning is **average length over long sequences**. Repetitions allow a code to spend short descriptions on common patterns and long descriptions on rare ones. Shannon's source-coding theorem says the optimal average length cannot beat entropy and can approach it for long blocks.

This laboratory illustrates the bound $nH$, not a full compression algorithm.

In [ ]:
itv.plot_repetition_code_lengths(probabilities=(0.5, 0.8, 0.95), n=120)

for p in [0.5, 0.8, 0.95, 0.99]:
    h = it.entropy([p, 1-p])
    print(f"P(1)={p:.2f}: H={h:.4f} bits/symbol; about {100*h:.1f} bits for 100 symbols")
print("A fixed-length binary representation always spends 100 bits on 100 binary symbols.")

# An actual sampled sequence: common events happen often but carry few bits each;
# rare events happen seldom but are individually surprising.
symbols = np.array(["common", "uncommon", "rare"])
p = np.array([0.90, 0.09, 0.01])
sequence = RNG.choice(symbols, size=20_000, p=p)
information = it.self_information(p)
counts = np.array([(sequence == symbol).sum() for symbol in symbols])
total_contribution = counts * information

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(symbols, counts, color="steelblue")
axes[0].set_title("Observed counts in 20,000 draws")
axes[0].set_ylabel("occurrences")
axes[1].bar(symbols, total_contribution, color="orange")
axes[1].set_title("Total information contributed by each outcome")
axes[1].set_ylabel("bits = count × self-information")
for ax in axes:
    ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

for symbol, probability, count, bits_each, total in zip(symbols, p, counts, information, total_contribution):
    print(f"{symbol:<8}: p={probability:.2f}, count={count:>5}, "
          f"{bits_each:.3f} bits/occurrence, {total:.1f} total bits")
print(f"Empirical average information = {total_contribution.sum()/len(sequence):.4f} bits/symbol")
print(f"Theoretical entropy            = {it.entropy(p):.4f} bits/symbol")

The fair source needs one bit per symbol because every position is maximally uncertain. A repetitive source has rare surprises, but most symbols are predictable; its average description length falls below one bit per symbol when encoded in blocks.

## 3. Entropy: Expected Self-Information

Before observing $X$, we do not know which self-information value will occur. Average over outcomes:

$$
\begin{aligned}
H(X)
&=E[I(X)]\\
&=\sum_x P(x)I(x)\\
&=-\sum_xP(x)\log_2P(x).
\end{aligned}
$$

$$\boxed{H(X)=-\sum_xP(x)\log_2P(x)}$$

The convention $0\log0=0$ follows from the limit $\lim_{p\to0^+}p\log p=0$: an impossible outcome never contributes to an expectation.

Entropy has two compatible readings:

* **Before observation:** expected uncertainty or surprise.
* **After many observations:** average information gained, and the theoretical minimum average bits needed to communicate the source.

Self-information $I(x)$ belongs to one outcome. Entropy $H(X)$ belongs to a distribution.

### Why the fair binary source maximizes entropy

For $X\sim Bernoulli(p)$:

$$H(p)=-p\log_2p-(1-p)\log_2(1-p).$$

Differentiate:

$$
H'(p)=\log_2\frac{1-p}{p}.
$$

$H'(p)=0$ implies $(1-p)/p=1$, so $p=1/2$. The second derivative is

$$H''(p)=-\frac{1}{\ln2}\left(\frac1p+\frac1{1-p}\right)<0,$$

therefore this stationary point is the unique maximum:

$$\boxed{H(1/2)=1\text{ bit}.}$$

More generally, among $K$ outcomes the uniform distribution maximizes entropy at $\log_2K$ bits. One proof uses KL non-negativity, derived later:

$$D_{KL}(P\parallel U)=\log_2K-H(P)\ge0.$$

Hence $H(P)\le\log_2K$, with equality only at $P=U$.

In [ ]:
distributions = {
    "Uniform": [0.25, 0.25, 0.25, 0.25],
    "Moderately concentrated": [0.55, 0.20, 0.15, 0.10],
    "Highly concentrated": [0.94, 0.02, 0.02, 0.02],
}
itv.plot_entropy_distributions(distributions)
print("Maximum for four outcomes = log2(4) =", np.log2(4), "bits")

# Trusted verification: scipy.stats.entropy uses the same definition.
for name, p in distributions.items():
    print(name, "scratch=", it.entropy(p), "SciPy=", stats.entropy(p, base=2))

p32 = np.full(32, 1/32)
print("\n32 equally likely outcomes:")
print("H =", it.entropy(p32), "bits; log2(32) =", np.log2(32), "bits")

# Progressively concentrate mass on outcome 0 while retaining the same 32 outcomes.
dominant = np.linspace(1/32, 0.99, 120)
h_path = []
for mass in dominant:
    distribution = np.full(32, (1-mass)/31)
    distribution[0] = mass
    h_path.append(it.entropy(distribution))
plt.figure(figsize=(8, 4))
plt.plot(dominant, h_path, color="purple", lw=2)
plt.axhline(5, color="black", ls="--", label=r"$\log_2 32=5$ bit maximum")
plt.xlabel("Probability assigned to one dominant outcome")
plt.ylabel("Entropy (bits)")
plt.title("On fixed support, increasing predictability reduces entropy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

The uniform distribution reaches the two-bit ceiling for four outcomes. Concentration makes the source more predictable and lowers entropy. Entropy is uncertainty of the **distribution**, not a count of outcomes alone.

Move the weights below. They are normalized automatically, so the widget always represents a valid probability distribution. Watch self-information rise for outcomes whose probability shrinks, while total entropy approaches the two-bit maximum near uniformity.

In [ ]:
itv.interactive_entropy()

## 4. Joint and Marginal Distributions

A marginal describes one variable alone. A joint distribution describes two variables together:

$$P(X=x,Y=y).$$

The table entries must be non-negative and sum to one. Summing rows or columns removes — *marginalizes out* — the other variable:

$$
P(X=x)=\sum_yP(x,y),\qquad
P(Y=y)=\sum_xP(x,y).
$$

Consider weather $X\in\{\text{sun},\text{rain}\}$ and an umbrella $Y\in\{\text{no},\text{yes}\}$:

$$
P(X,Y)=
\begin{bmatrix}
0.45&0.05\\
0.10&0.40
\end{bmatrix}.
$$

For example, $P(X=\text{rain})=0.10+0.40=0.50$ and $P(Y=\text{yes})=0.05+0.40=0.45$.

Joint entropy treats each pair $(x,y)$ as an outcome:

$$
\boxed{H(X,Y)=-\sum_x\sum_yP(x,y)\log_2P(x,y).}
$$

In [ ]:
weather_umbrella = np.array([[0.45, 0.05], [0.10, 0.40]])
px, py = it.marginals(weather_umbrella)
print("P(X) =", px, "P(Y) =", py)
print("H(X,Y) =", it.joint_entropy(weather_umbrella), "bits")
itv.plot_joint_table(
    weather_umbrella,
    x_labels=["sun", "rain"],
    y_labels=["no umbrella", "umbrella"],
    title="Weather and umbrella",
)

The marginals alone cannot reconstruct the heatmap: they say how common rain and umbrellas are, but not whether they occur together. Dependence lives in the joint distribution.

## 5. Conditional Probability and Conditional Entropy

Conditioning means restricting attention to rows compatible with what we learned:

$$P(Y=y\mid X=x)=\frac{P(x,y)}{P(x)}.$$

For rain,

$$P(\text{umbrella}\mid\text{rain})=\frac{0.40}{0.50}=0.80.$$

For one known value $X=x$, remaining uncertainty is

$$H(Y\mid X=x)=-\sum_yP(y\mid x)\log_2P(y\mid x).$$

Before observing $X$, we do not know which row will be revealed, so average with $P(x)$:

$$
\boxed{
H(Y\mid X)=\sum_xP(x)H(Y\mid X=x)
=-\sum_{x,y}P(x,y)\log_2P(y\mid x)
}
$$

Conditional entropy is generally asymmetric: $H(Y\mid X)\neq H(X\mid Y)$.

Three limiting cases build the intuition:

* **Independent:** $P(y\mid x)=P(y)$, so $H(Y\mid X)=H(Y)$. Knowing $X$ helps nothing.
* **Identical/deterministic:** $Y=f(X)$, so $H(Y\mid X)=0$. Knowing $X$ removes all uncertainty.
* **Partial/noisy dependence:** $0<H(Y\mid X)<H(Y)$.

In [ ]:
independent = np.array([[0.25, 0.25], [0.25, 0.25]])
deterministic = np.array([[0.50, 0.00], [0.00, 0.50]])
partial = np.array([[0.40, 0.10], [0.10, 0.40]])
cases = {"Independent": independent, "Deterministic": deterministic, "Partial": partial}
itv.plot_conditional_cases(cases)

for name, joint in cases.items():
    h_y = it.entropy(joint.sum(axis=0))
    h_cond = it.conditional_entropy(joint)
    print(f"{name:<14}: H(Y)={h_y:.3f}, H(Y|X)={h_cond:.3f}")

## 6. Chain Rule of Entropy

Probability factorizes:

$$P(x,y)=P(x)P(y\mid x).$$

Insert that factorization into joint entropy:

$$
\begin{aligned}
H(X,Y)
&=-\sum_{x,y}P(x,y)\log_2P(x,y)\\
&=-\sum_{x,y}P(x,y)\log_2[P(x)P(y\mid x)]\\
&=-\sum_{x,y}P(x,y)\log_2P(x)
  -\sum_{x,y}P(x,y)\log_2P(y\mid x).
\end{aligned}
$$

In the first term, $\sum_yP(x,y)=P(x)$, so it becomes $H(X)$. The second term is the definition of $H(Y\mid X)$:

$$
\boxed{H(X,Y)=H(X)+H(Y\mid X).}
$$

Factorizing in the other direction also gives

$$\boxed{H(X,Y)=H(Y)+H(X\mid Y).}$$

The identity is a bookkeeping law: uncertainty about the pair equals uncertainty about the first variable plus what remains about the second after the first is known.

In [ ]:
joint = weather_umbrella
h_joint = it.joint_entropy(joint)
h_x = it.entropy(joint.sum(axis=1))
h_y = it.entropy(joint.sum(axis=0))
h_y_x = it.conditional_entropy(joint, target="y")
h_x_y = it.conditional_entropy(joint, target="x")
print(f"H(X,Y)              = {h_joint:.12f}")
print(f"H(Y)                = {h_y:.12f}")
print(f"H(X)                = {h_x:.12f}")
print(f"H(Y|X)              = {h_y_x:.12f}")
print(f"H(X|Y)              = {h_x_y:.12f}")
print(f"H(X) + H(Y|X)       = {h_x + h_y_x:.12f}")
print(f"H(Y) + H(X|Y)       = {h_y + h_x_y:.12f}")
assert np.isclose(h_joint, h_x + h_y_x)
assert np.isclose(h_joint, h_y + h_x_y)

## 7. Mutual Information: Uncertainty Removed

Before learning $X$, uncertainty about $Y$ is $H(Y)$. After learning $X$, it is $H(Y\mid X)$. Their difference is the information $X$ supplies about $Y$:

$$\boxed{I(X;Y)=H(Y)-H(Y\mid X).}$$

Use the chain rule $H(X,Y)=H(X)+H(Y\mid X)$:

$$
\begin{aligned}
I(X;Y)
&=H(Y)-[H(X,Y)-H(X)]\\
&=\boxed{H(X)+H(Y)-H(X,Y)}.
\end{aligned}
$$

That expression is symmetric, therefore

$$I(X;Y)=I(Y;X),$$

even though conditional entropies are not generally symmetric.

Substituting definitions gives the probability-ratio form:

$$
\boxed{
I(X;Y)=\sum_{x,y}P(x,y)\log_2\frac{P(x,y)}{P(x)P(y)}
}.
$$

The denominator $P(x)P(y)$ is the joint distribution we would see under independence. MI therefore measures departure from independence. This formula can be proved by substituting P(X) by the formula stated in the conditional distribution and then using logarithm and bayes rule. 

### Why MI is non-negative

The ratio form is a KL divergence:

$$I(X;Y)=D_{KL}(P_{XY}\parallel P_XP_Y)\ge0.$$

A short proof of KL non-negativity uses $\ln u\le u-1$. Set $u=q_i/p_i$:

$$
-D_{KL}(P\parallel Q)
=\sum_i p_i\ln\frac{q_i}{p_i}
\le\sum_i p_i\left(\frac{q_i}{p_i}-1\right)
=\sum_iq_i-\sum_ip_i=0.
$$

Thus $D_{KL}\ge0$, with equality only when $P=Q$ on the support. For discrete variables:

$$I(X;Y)=0\iff P(x,y)=P(x)P(y),$$

so zero MI corresponds to independence.

In [ ]:
for name, joint in cases.items():
    mi_ratio = it.mutual_information(joint)
    mi_reduction = it.entropy(joint.sum(axis=0)) - it.conditional_entropy(joint)
    mi_symmetric = it.entropy(joint.sum(axis=1)) + it.entropy(joint.sum(axis=0)) - it.joint_entropy(joint)
    print(f"{name:<14}: ratio={mi_ratio:.6f}, reduction={mi_reduction:.6f}, symmetric={mi_symmetric:.6f}")

## 8. Mutual-Information Laboratory: Linear, Nonlinear, and Noisy Dependence

For continuous variables we do not have an exact finite probability table. The plots below use scikit-learn's $k$-nearest-neighbor MI **estimator**, reported in nats. It is sample- and tuning-dependent; it is not the exact discrete sum above.

Pearson correlation measures linear association. MI detects any statistical dependence that the estimator can resolve. Therefore:

* independence $\Rightarrow$ zero correlation and zero MI
* zero correlation $\not\Rightarrow$ independence
* in the symmetric quadratic relation $Y=X^2+\epsilon$, positive and negative $X$ cancel in Pearson's linear covariance even though $Y$ depends strongly on $X$.

In [ ]:
itv.plot_dependence_laboratory(noise=0.20)
itv.plot_mi_vs_noise(kind="quadratic")

The quadratic and sinusoidal panels are the memorable counterexample: $r\approx0$ while MI is positive. Increasing noise blurs the mapping and reduces MI. The estimate may wiggle rather than decrease perfectly because it is itself a statistic computed from a finite sample.

Use the explorer to switch relationship type and noise. Do not compare the numerical magnitude of Pearson $r$ and MI as if they shared units; compare whether each detects structure.

In [ ]:
itv.interactive_mutual_information()

## 9. Real Language as an Information Source

We now ask: **how is information distributed in real written language?**

### Data and scope

The bundled UTF-8 files are static snapshots of the Universal Declaration of Human Rights from `wooorm/udhr`, ultimately sourced from OHCHR / Unicode UDHR:

* English (`eng`)
* Amharic (`amh`, Ethiopic script)
* Spanish (`spa`)

They were downloaded on 2026-09-13. The packaging project is MIT licensed; source attribution and underlying OHCHR/Unicode terms are documented in `data/information_theory/README.md`.

This is one document per language, not a balanced corpus and not evidence that one language is universally “more complex.”

### What is a symbol?

We normalize Unicode to NFC, lowercase where the script supports it, retain letters/combining marks/digits, collapse all whitespace to one ordinary space, and remove punctuation. We **do not transliterate** Amharic or force it through an English alphabet. A Unicode code point after this preprocessing is one symbol.

Changing those choices changes the probability distribution and therefore changes entropy. Spaces, punctuation, normalization, corpus genre, and sample length are part of the model.

In [ ]:
language_files = {
    "English": DATA_DIR / "udhr_eng.txt",
    "Amharic": DATA_DIR / "udhr_amh.txt",
    "Spanish": DATA_DIR / "udhr_spa.txt",
}
raw_texts = {name: path.read_text(encoding="utf-8") for name, path in language_files.items()}
texts = {name: it.preprocess_text(text, lowercase=True, keep_spaces=True, keep_punctuation=False)
         for name, text in raw_texts.items()}

#print(texts["Amharic"])

language_stats = {}
for name, text in texts.items():
    language_stats[name] = {
        "characters": it.character_information(text),
        "adjacent": it.adjacent_character_information(text),
    }
    print(f"{name:<8}: raw={len(raw_texts[name]):>6}, processed={len(text):>6}, "
          f"alphabet={len(set(text)):>3}", f"Entropy = {language_stats[name]["characters"]["entropy"]}")

itv.plot_language_statistics(language_stats)

Character entropy $H(C)$ describes the uncertainty of one symbol drawn without context. Conditional entropy $H(C_{i+1}\mid C_i)$ is lower because spelling and whitespace create sequential structure. Their difference is adjacent-character MI:

$$I(C_i;C_{i+1})=H(C_{i+1})-H(C_{i+1}\mid C_i).$$

We also verify

$$I(C_i;C_{i+1})=H(C_i)+H(C_{i+1})-H(C_i,C_{i+1}).$$

Differences across languages reflect these specific texts, scripts, and preprocessing decisions. They are not universal complexity rankings.

In [ ]:
print(f"{'Language':<10} {'H(C)':>9} {'H(next|current)':>17} {'MI adjacent':>13} {'identity gap':>13}")
for name, item in language_stats.items():
    ch, adj = item["characters"], item["adjacent"]
    print(f"{name:<10} {ch['entropy']:9.3f} {adj['H_next_given_current']:17.3f} "
          f"{adj['MI_adjacent']:13.3f} {abs(adj['MI_adjacent']-adj['MI_identity']):13.2e}")

# Small preprocessing sensitivity check: punctuation retained versus removed.
for name, raw in raw_texts.items():
    no_punct = it.preprocess_text(raw, keep_punctuation=False)
    with_punct = it.preprocess_text(raw, keep_punctuation=True)
    print(f"{name}: H without punctuation={it.character_information(no_punct)['entropy']:.3f}; "
          f"with punctuation={it.character_information(with_punct)['entropy']:.3f}")

### Language identification through cross-entropy

We do not need a sophisticated classifier to see an information-theoretic signal. Build a smoothed character distribution $Q_\ell$ for each language. Given a held-out sample, compute its empirical distribution $P_{\text{sample}}$ and the cross-entropy under each $Q_\ell$:

$$H(P_{\text{sample}},Q_\ell)=-\sum_cP_{\text{sample}}(c)\log_2Q_\ell(c).$$

The smallest value is the language model that spends the fewest average bits describing the sample. Add-one smoothing prevents unseen characters from receiving zero probability.

In [ ]:
alphabet = sorted(set().union(*(set(t) for t in texts.values())))

def smoothed_model(text, alpha=1.0):
    counts = {c: alpha for c in alphabet}
    for c in text:
        counts[c] += 1
    total = sum(counts.values())
    return np.array([counts[c] / total for c in alphabet])

models = {name: smoothed_model(text[:int(0.8*len(text))]) for name, text in texts.items()}
print("Held-out cross-entropy in bits/character (lower is more compatible)")
print(f"{'sample':<10}" + "".join(f"{name:>12}" for name in models))
for sample_name, text in texts.items():
    held_out = text[int(0.8*len(text)):]
    counts = np.array([held_out.count(c) for c in alphabet], dtype=float)
    p_sample = counts / counts.sum()
    costs = {name: it.cross_entropy(p_sample, q) for name, q in models.items()}
    print(f"{sample_name:<10}" + "".join(f"{costs[name]:12.3f}" for name in models))
print("The diagonal should usually be smallest; this is compatibility, not a full NLP system.")

## 10. Substitution Cipher: What Information Survives?

A monoalphabetic substitution applies a one-to-one relabeling of symbols. It hides identity but preserves:

* the multiset of single-character counts
* character entropy
* relabeled bigram/joint structure
* conditional entropy and adjacent-character MI

It does **not** preserve readable character names. Frequency analysis exploits exactly this surviving structure. Entropy alone does not uniquely decrypt a message, and none of this applies to claiming attacks on modern cryptography such as AES.

In [ ]:
plain = texts["English"]
cipher, mapping = it.monoalphabetic_substitution(plain, seed=7)
itv.plot_plaintext_cipher_comparison(plain, cipher)

p_char, c_char = it.character_information(plain), it.character_information(cipher)
p_adj, c_adj = it.adjacent_character_information(plain), it.adjacent_character_information(cipher)
print("Plain/cipher character entropy:", p_char["entropy"], c_char["entropy"])
print("Plain/cipher conditional entropy:", p_adj["H_next_given_current"], c_adj["H_next_given_current"])
print("Plain/cipher adjacent MI:", p_adj["MI_adjacent"], c_adj["MI_adjacent"])
print("First 120 plaintext symbols :", plain[:120])
print("First 120 encrypted symbols :", cipher[:120])

Two sequences can also have the same marginal frequencies and radically different order. Alternating `0101...` and a shuffled sequence with 50% zeros both have $H(C)\approx1$ bit, but the alternating next symbol is deterministic: $H(C_{i+1}\mid C_i)=0$. Marginal entropy alone does not capture sequence structure.

In [ ]:
alternating = "01" * 1000
shuffled = list(alternating)
RNG.shuffle(shuffled)
shuffled = "".join(shuffled)
for name, sequence in {"alternating": alternating, "shuffled": shuffled}.items():
    ch = it.character_information(sequence)
    adj = it.adjacent_character_information(sequence)
    print(f"{name:<12}: H(C)={ch['entropy']:.4f}, H(next|current)={adj['H_next_given_current']:.4f}, "
          f"I(adjacent)={adj['MI_adjacent']:.4f}")

## 11. Football Market-Value Information Laboratory

As a football fan might ask: **which player attributes contain the most information about market value?**

### Reproducible snapshot

The bundled CSV is a deterministic 12,000-player snapshot prepared from the open `dcaribou/transfermarkt-datasets` `players` and `appearances` tables on 2026-09-13. It contains current market value, age, profile attributes, and appearance statistics aggregated over the source's observed games. No live API is needed.

For this experiment we restrict the population to roles for which goals and assists are meaningful attacking outputs:

* centre-forward
* second striker
* left or right winger
* attacking midfielder

Goalkeepers and defenders are excluded. Comparing their prices with goals would mostly rediscover role conventions: a goalkeeper is not expected to score. Clean sheets would be appropriate for goalkeepers and defenders, but the source appearance table does not contain clean-sheet or goals-conceded fields, so filtering is more defensible than inventing an unavailable metric.

Target:

$$Y=\log(1+\text{market value in EUR}).$$

The log reduces extreme right-skew; it does not turn MI into causality.

### Features and estimators

Numerical candidates include age, height, appearances, minutes, total goals, total assists, goals per 90, assists per 90, and cards. Totals measure production mixed with opportunity; per-90 rates partly separate scoring/creation rate from playing time. To avoid absurd rates from tiny cameos, per-90 values are calculated only after at least 450 observed minutes (five full matches). Categorical candidates include attacking role, foot, citizenship, and club.

For numerical variables we use scikit-learn's $k$-NN `mutual_info_regression` estimator. For categorical variables we ordinal-encode categories and mark the feature as discrete. These are **estimates in nats**, not exact probability-table MI. High-cardinality variables such as club can also receive optimistic finite-sample estimates.

Before calculation, a reasonable prediction is that age, performance exposure, goals/assists, position, and club context contain information; foot and height may be weaker. The data, not the story, gets the final word.

In [ ]:
football_all = pd.read_csv( DATA_DIR / "football_players_snapshot.csv" )
attacking_roles = ["Centre-Forward", "Second Striker", "Left Winger", "Right Winger", "Attacking Midfield"]

# Keep only attacking players
football = football_all[ football_all["sub_position"].isin(attacking_roles)].copy()

# Require at least five full matches for per-90 statistics
rate_eligible = football["minutes_played"] >= 450
football["goals_per_90"] = np.where( rate_eligible, 90 * football["goals"] / football["minutes_played"], np.nan)
football["assists_per_90"] = np.where(rate_eligible, 90 * football["assists"] / football["minutes_played"], np.nan)

# Log-transform market value to reduce extreme skew
football["log_market_value"] = np.log1p( football["market_value_in_eur"])

print(f"Full snapshot: {len(football_all):,} players")
print(f"Attacking-role analysis set: {len(football):,} players")

print("\nPlayers by attacking position:")
print(football["sub_position"].value_counts())

print("\nMarket value median / max:")
print(football["market_value_in_eur"].median(), football["market_value_in_eur"].max(),)

display(football.head(3))


numeric_features = ["age","height_in_cm","appearances","minutes_played","goals","assists","goals_per_90","assists_per_90","yellow_cards","red_cards"]
categorical_features = ["sub_position","foot","country_of_citizenship","current_club_id"]
target = "log_market_value"

rows = []

for feature in numeric_features:
    # Drop rows jointly so X and Y always refer to exactly the same players.
    data = football[[feature, target]].dropna()

    x = data[feature].to_numpy().reshape(-1, 1)
    y = data[target].to_numpy()

    # Mutual information
    mi = mutual_info_regression(x,y,random_state=42)[0]

    # Spearman rank correlation
    spearman = abs(stats.spearmanr(x[:, 0],y).statistic)

    rows.append({
        "feature": feature,
        "MI_nats": mi,
        "abs_spearman": spearman,
        "n": len(data),
        "method": "continuous kNN",
    })

for feature in categorical_features:
    
    # Again, remove missing rows jointly.
    data = football[[feature, target]].dropna()

    values = data[[feature]].astype(str)
    y = data[target].to_numpy()

    # Convert categories to integer codes.
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1,)
    encoded = encoder.fit_transform(values)

    # MI estimator treats this feature as discrete.
    mi = mutual_info_regression(encoded,y,discrete_features=True,random_state=42)[0]

    rows.append({
        "feature": feature,
        "MI_nats": mi,
        "abs_spearman": np.nan,
        "n": len(data),
        "method": "discrete-category estimator",
    })


mi_rank = (pd.DataFrame(rows).sort_values("MI_nats",ascending=False).reset_index(drop=True))
display(mi_rank)

fig, axes = plt.subplots(2,2,figsize=(14, 10))
plot_data = mi_rank.sort_values("MI_nats")

axes[0, 0].barh(plot_data["feature"],plot_data["MI_nats"])
axes[0, 0].set_title("Estimated information about log market value")
axes[0, 0].set_xlabel("Mutual information (nats)")
axes[0, 0].set_ylabel("Feature")


numeric_rank = mi_rank.dropna(subset=["abs_spearman"]).copy()

ax = axes[0, 1]
ax.scatter(numeric_rank["abs_spearman"],numeric_rank["MI_nats"],s=70)

# Label every feature
texts = []

for _, row in numeric_rank.iterrows():

    text = ax.annotate(row["feature"],
            (
                row["abs_spearman"],
                row["MI_nats"],
            ),
            xytext=(6, 6),
            textcoords="offset points",
        )

    texts.append(text)


ax.set_xlabel("|Spearman correlation|")
ax.set_ylabel("Mutual information (nats)")
ax.set_title("Monotone association versus general dependence")
ax.grid(True,alpha=0.3)

positive_minutes = (football["minutes_played"] > 0)

for ax, feature, title in [(axes[1, 0],"goals","Career goals versus market value"),(axes[1, 1],"assists","Career assists versus market value")]:

    plot_data = football.loc[positive_minutes,[feature, "log_market_value"]].dropna()

    hb = ax.hexbin(np.log1p(plot_data[feature]),plot_data["log_market_value"],gridsize=35,mincnt=1,cmap="Blues",)
    
    ax.set_xlabel(f"log(1 + {feature})")
    ax.set_ylabel("log(1 + market value EUR)")
    ax.set_title(title + "\n(attacking roles only)")
    fig.colorbar(hb,ax=ax,label="players per hexagon")
    ax.grid(True,alpha=0.3)

plt.tight_layout()
plt.show()

Interpret the ranking as a property of this snapshot and estimator. Appearances/minutes can encode opportunity, career length, league coverage, and selection. Club can proxy reputation, league quality, resources, and ability. None of those statements proves a causal effect:

$$I(X;Y)>0\not\Rightarrow X\text{ causes }Y.$$

Potential confounders include ability, age, league quality, exposure, reputation, and selection into clubs. Total goals and assists also mix rate with opportunity: a player must receive minutes to accumulate either. The per-90 features help inspect that distinction, but they can be unstable for players with very few minutes and still do not establish causality.

### Discretization sensitivity

An alternative is to quantile-bin both a feature and value, then calculate exact discrete MI on the resulting table. The arithmetic is exact **for that discretized table**, but the answer depends on bin count. Discretization is a modeling choice, not harmless preprocessing.

In [ ]:
feature = football["age"].to_numpy()
target = football["log_market_value"].to_numpy()
for bins in [3, 5, 8, 12]:
    xb, _ = it.quantile_bins(feature, bins)
    yb, _ = it.quantile_bins(target, bins)
    _, _, joint = it.joint_distribution(xb, yb)
    print(f"{bins:>2} quantile bins: exact MI of discretized table = {it.mutual_information(joint):.4f} bits")
print("Changing bins changes the question and the estimated information.")

## 12. Information Gain in a Decision-Tree Split

A decision-tree split observes a feature $X$ to reduce uncertainty about a class label $Y$:

$$
\boxed{IG(Y,X)=H(Y)-H(Y\mid X)=I(X;Y).}
$$

We inspect one node deeply rather than building a tree library. The target is whether a football player is above the median market value. Candidate binary splits ask whether the player is young, has many appearances, has many goals, or is left-footed.

For every candidate we calculate:

1. parent class probabilities and $H(Y)$
2. each branch's class distribution
3. weighted child entropy $H(Y\mid X)$
4. information gain

The selected feature is

$$X^*=\arg\max_X IG(Y,X).$$

In [ ]:
tree_data = football.copy()
tree_data["high_value"] = (tree_data["market_value_in_eur"] > tree_data["market_value_in_eur"].median()).astype(int)
candidate_splits = {
    "age <= 24": (tree_data["age"] <= 24).astype(int),
    "appearances >= median": (tree_data["appearances"] >= tree_data["appearances"].median()).astype(int),
    "goals >= median": (tree_data["goals"] >= tree_data["goals"].median()).astype(int),
    "left foot": (tree_data["foot"].fillna("unknown") == "left").astype(int),
}
ig_results = {name: it.information_gain(values, tree_data["high_value"]) for name, values in candidate_splits.items()}

print(f"{'candidate':<24} {'H(Y)':>8} {'H(Y|X)':>10} {'IG':>9} {'MI check':>10}")
for name, result in ig_results.items():
    print(f"{name:<24} {result['H_target']:8.4f} {result['H_target_given_feature']:10.4f} "
          f"{result['information_gain']:9.4f} {result['mutual_information']:10.4f}")
    branch_totals = result["joint"].sum(axis=1, keepdims=True)
    branch_classes = np.divide(
        result["joint"], branch_totals,
        out=np.zeros_like(result["joint"]), where=branch_totals > 0,
    )
    print("  branch class probabilities [low value, high value]:", branch_classes)
best = max(ig_results, key=lambda n: ig_results[n]["information_gain"])
print("\nBest from-scratch split:", best)
itv.plot_information_gain(ig_results)

# Trusted check: sklearn's depth-one entropy tree chooses among these columns.
X_tree = pd.DataFrame(candidate_splits)
clf = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=42).fit(X_tree, tree_data["high_value"])
print("sklearn root feature:", X_tree.columns[clf.tree_.feature[0]])
assert X_tree.columns[clf.tree_.feature[0]] == best

The parent entropy is identical for every candidate because the target has not changed. A useful split creates purer branches, which lowers weighted child entropy. The difference is the exact number of bits the split removes on average at this node.

## 13. Cross-Entropy: When a Model Supplies the Probabilities

Entropy uses the true distribution $P$ both to generate outcomes and assign their code lengths:

$$H(P)=-\sum_xP(x)\log P(x).$$

Suppose reality still follows $P$, but a model $Q$ assigns the probabilities. If $x$ occurs, the model's information cost is $-\log Q(x)$. Average under what actually occurs:

$$
\boxed{H(P,Q)=-\sum_xP(x)\log Q(x).}
$$

* $P$ decides how often outcomes occur.
* $Q$ decides how many bits/nats the model assigns.
* Cross-entropy is the expected cost of describing $P$ using $Q$.

Assigning tiny probability to an outcome with substantial true probability is expensive because $-\log q\to\infty$ as $q\to0$.

In [ ]:
itv.plot_cross_entropy_penalty()
itv.interactive_cross_entropy()

### Equal accuracy, different probabilistic quality

Accuracy discards confidence: it only asks which side of $0.5$ a prediction lies on. Cross-entropy asks how much probability was assigned to what happened.

The two artificial models below make exactly the same class decisions. Model B is extremely confident when it is wrong, so its negative log cost is much larger.

In [ ]:
y = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
# Same predicted classes: indices 3 and 7 are wrong for both.
q_a = np.array([.72, .28, .68, .42, .25, .31, .76, .58, .66, .33])
q_b = np.array([.97, .03, .95, .001, .02, .04, .98, .999, .93, .05])
itv.plot_equal_accuracy_models(y, q_a, q_b)
for name, q in {"A": q_a, "B": q_b}.items():
    accuracy = np.mean((q >= .5) == y)
    scratch = np.mean(it.binary_cross_entropy(y, q))
    library = log_loss(y, q)
    print(f"Model {name}: accuracy={accuracy:.2f}, scratch CE={scratch:.5f}, sklearn={library:.5f}")

## 14. Categorical and Binary Cross-Entropy

For one-hot target $y=(y_1,\ldots,y_K)$ and predicted probabilities $\hat y$:

$$L=-\sum_{k=1}^Ky_k\log\hat y_k.$$

If the true class is $j$, then $y_j=1$ and every other $y_k=0$:

$$
L=-[0\log\hat y_1+\cdots+1\log\hat y_j+\cdots+0\log\hat y_K]
=\boxed{-\log\hat y_j}.
$$

One-hot encoding simply selects the probability assigned to the correct class. The loss is the self-information that the model assigned to the outcome that occurred.

Models often output logits $z_k$. Softmax creates a valid distribution:

$$\hat y_k=\frac{e^{z_k}}{\sum_je^{z_j}}.$$

Subtracting $\max z$ before exponentiation changes neither ratio and prevents overflow.

For binary $y\in\{0,1\}$ and $\hat y=P(Y=1)$:

$$
\boxed{L=-y\log\hat y-(1-y)\log(1-\hat y).}
$$

If $y=1$, the second term vanishes and $L=-\log\hat y$. If $y=0$, the first vanishes and $L=-\log(1-\hat y)$. In both cases the loss is negative log probability of the actual outcome.

In [ ]:
logits = np.array([1.2, -0.4, 2.1])
probabilities = it.softmax(logits)
one_hot = np.array([0, 0, 1])
print("softmax probabilities:", probabilities, "sum =", probabilities.sum())
print("categorical CE:", it.categorical_cross_entropy(one_hot, probabilities))
print("-log probability of true class:", -np.log(probabilities[2]))
itv.plot_binary_cross_entropy()

## 15. KL Divergence: The Price of the Wrong Distribution

Subtract true entropy from cross-entropy:

$$
\begin{aligned}
H(P,Q)-H(P)
&=-\sum_xP(x)\log Q(x)+\sum_xP(x)\log P(x)\\
&=\sum_xP(x)\log\frac{P(x)}{Q(x)}\\
&=\boxed{D_{KL}(P\parallel Q)}.
\end{aligned}
$$

Therefore

$$\boxed{H(P,Q)=H(P)+D_{KL}(P\parallel Q).}$$

Because $D_{KL}\ge0$, using the wrong distribution cannot beat the ideal average code based on $P$:

$$H(P,Q)\ge H(P).$$

Interpretation:

* $H(P)$: unavoidable uncertainty inherent in reality.
* $D_{KL}(P\parallel Q)$: additional cost due to model mismatch.

When optimizing $Q$, $H(P)$ is constant. Minimizing cross-entropy is therefore equivalent to minimizing $D_{KL}(P\parallel Q)$.

KL is asymmetric and is not a distance metric: generally

$$D_{KL}(P\parallel Q)\neq D_{KL}(Q\parallel P).$$

In [ ]:
p = np.array([0.60, 0.25, 0.10, 0.05])
q_models = {
    "Q=P": p.copy(),
    "moderate": np.array([0.45, 0.30, 0.15, 0.10]),
    "poor": np.array([0.15, 0.20, 0.25, 0.40]),
}
metrics = itv.plot_kl_experiment(p, q_models)
print(f"{'model':<12} {'H(P)':>9} {'H(P,Q)':>10} {'KL(P||Q)':>11} {'identity gap':>13}")
for name, h, ce, kl in metrics:
    print(f"{name:<12} {h:9.5f} {ce:10.5f} {kl:11.5f} {abs(ce-h-kl):13.2e}")

q = q_models["poor"]
print("\nAsymmetry check")
print("KL(P||Q) =", it.kl_divergence(p, q))
print("KL(Q||P) =", it.kl_divergence(q, p))

## 16. Cross-Entropy Is Negative Log-Likelihood

Notebook 04 derived likelihood for independent observations:

$$L(Q;x_1,\ldots,x_n)=\prod_{i=1}^nQ(x_i).$$

Take negative logarithms:

$$
-\log L=-\sum_{i=1}^n\log Q(x_i).
$$

Divide by $n$:

$$
-\frac1n\log L
=-\sum_x\hat P_n(x)\log Q(x)
=H(\hat P_n,Q),
$$

where $\hat P_n$ is the empirical frequency distribution. Average negative log-likelihood is empirical cross-entropy. This loss is not an arbitrary neural-network convention; it emerges from maximum likelihood for probabilistic classification.

In [ ]:
observations = np.array([0, 0, 0, 1, 0, 1, 0, 0, 1, 0])
q = np.array([0.7, 0.3])
likelihood = np.prod(q[observations])
average_nll = -np.log(likelihood) / len(observations)
counts = np.bincount(observations, minlength=2)
p_empirical = counts / counts.sum()
empirical_ce = it.cross_entropy(p_empirical, q, base=np.e)
print("Likelihood =", likelihood)
print("Average negative log-likelihood =", average_nll)
print("Empirical cross-entropy =", empirical_ce)
assert np.isclose(average_nll, empirical_ce)

## 17. Final Synthesis

1. Rare events carry more self-information because observing them changes our expectations more.
2. Self-information is $I(x)=-\log_2P(x)$; one of eight equally likely outcomes carries three bits.
3. The logarithm makes information from independent events additive.
4. Entropy is expected self-information: $H(X)=E[I(X)]$.
5. Uniform distributions maximize entropy on a fixed support: $H(X)\le\log_2K$; 32 equal outcomes require five bits on average.
6. Marginal distributions describe variables separately; the joint describes $(X,Y)$ together.
7. Joint entropy applies entropy to pair outcomes: $H(X,Y)=-\sum_{x,y}P(x,y)\log P(x,y)$.
8. Conditional entropy $H(Y\mid X)$ is uncertainty that remains after learning $X$.
9. Probability factorization produces the chain rule $H(X,Y)=H(X)+H(Y\mid X)$.
10. Under independence, knowing $X$ helps nothing: $H(Y\mid X)=H(Y)$.
11. Under a deterministic relationship, knowing $X$ can remove all uncertainty: $H(Y\mid X)=0$.
12. Mutual information is uncertainty removed: $I(X;Y)=H(Y)-H(Y\mid X)$.
13. The chain rule gives the symmetric identity $I(X;Y)=H(X)+H(Y)-H(X,Y)$.
14. MI is symmetric and non-negative; exact discrete MI is zero exactly at independence.
15. Zero Pearson correlation does not imply independence.
16. MI can reveal nonlinear dependence that linear correlation misses.
17. Decision-tree information gain is MI-based uncertainty reduction at a node.
18. Real language contains sequential information beyond single-character frequencies.
19. Conditional character entropy and adjacent-character MI quantify that sequential structure.
20. A simple substitution cipher preserves relabeled frequency and adjacency structure, but entropy alone does not decrypt it.
21. Attacker-only football MI estimates which features contain information about market value.
22. Statistical information is not causality; ability, exposure, age, league, and selection can confound the relationships.
23. Cross-entropy is the expected cost when reality follows $P$ but a model assigns probabilities $Q$.
24. Confidently assigning tiny probability to the true outcome creates a very large logarithmic penalty.
25. Equal accuracy can conceal very different cross-entropy and probabilistic quality.
26. One-hot categorical cross-entropy selects $-\log\hat y_{\text{true}}$.
27. Binary cross-entropy selects $-\log\hat y$ for $y=1$ and $-\log(1-\hat y)$ for $y=0$.
28. $H(P,Q)=H(P)+D_{KL}(P\parallel Q)$ separates unavoidable uncertainty from model mismatch.
29. Minimizing cross-entropy in $Q$ minimizes $D_{KL}(P\parallel Q)$ because $H(P)$ is fixed; KL is asymmetric and not a metric.
30. Average negative log-likelihood is empirical cross-entropy, so classification loss follows from probabilistic modeling.

The coherent picture:

$$
\boxed{
\text{Information theory measures uncertainty, surprise, dependence, information gained,
and the cost of assigning incorrect probabilities.}
}
$$